## Stage 0 — World and LLM Setup

Loads the PR2-apartment world fixture and creates the LLM.

`groundable_type` scopes entity grounding and world serialisation to physical bodies.
It defaults to `Symbol` (all instances) inside `llmr` — passing `Body` gives
a tighter, physical-body-only scope. It is **never** imported inside the library itself.


In [1]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()

# Pass Body for physical-body-scoped grounding (tighter than the Symbol default).
groundable_type = Body

print(f"groundable_type : {groundable_type}")
print(f"World loaded.  Scene bodies in SymbolGraph:")
from krrood.symbol_graph.symbol_graph import SymbolGraph
for inst in SymbolGraph().get_instances_of_type(Body):
    from llmr.world.serializer import body_display_name
    print(f"  {body_display_name(inst)}")

INSTRUCTION = "pick up the milk from the table"
print(f"\nInstruction: {INSTRUCTION!r}")


`polytope` failed to import `cvxopt.glpk`.
will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


groundable_type : <class 'semantic_digital_twin.world_description.world_entity.Body'>
World loaded.  Scene bodies in SymbolGraph:
  base_link
  base_footprint
  base_bellow_link
  base_laser_link
  fl_caster_rotation_link
  fl_caster_l_wheel_link
  fl_caster_r_wheel_link
  fr_caster_rotation_link
  fr_caster_l_wheel_link
  fr_caster_r_wheel_link
  bl_caster_rotation_link
  bl_caster_l_wheel_link
  bl_caster_r_wheel_link
  br_caster_rotation_link
  br_caster_l_wheel_link
  br_caster_r_wheel_link
  torso_lift_link
  l_torso_lift_side_plate_link
  r_torso_lift_side_plate_link
  torso_lift_motor_screw_link
  imu_link
  head_pan_link
  head_tilt_link
  head_plate_frame
  sensor_mount_link
  high_def_frame
  high_def_optical_frame
  double_stereo_link
  wide_stereo_link
  wide_stereo_optical_frame
  wide_stereo_l_stereo_camera_frame
  wide_stereo_l_stereo_camera_optical_frame
  wide_stereo_r_stereo_camera_frame
  wide_stereo_r_stereo_camera_optical_frame
  narrow_stereo_link
  narrow_stereo_

In [2]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [3]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [4]:
# ── LLM ───────────────────────────────────────────────────────────────────────
# Swap provider/model freely — everything downstream accepts any BaseChatModel.
from dotenv import load_dotenv
from llmr.reasoning.llm_config import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print(f"LLM ready: {llm.model_name}")

LLM ready: gpt-4o-mini


## Stage 1 — Discover Action Classes

`discover_action_classes()` (from `llmr.pycram_bridge`) scans
`pycram.robot_plans.actions` and returns every concrete class whose name ends
with `"Action"`.  All pycram access is funnelled through `pycram_bridge` —
no direct pycram import elsewhere in llmr.


In [5]:
from llmr.pycram_bridge import discover_action_classes

action_registry = discover_action_classes()

print(f"Discovered {len(action_registry)} action classes:\n")
for name, cls in sorted(action_registry.items()):
    print(f"  {name:30s}  {cls.__module__}.{cls.__qualname__}")


Discovered 24 action classes:

  CarryAction                     pycram.robot_plans.actions.core.robot_body.CarryAction
  CloseAction                     pycram.robot_plans.actions.core.container.CloseAction
  CuttingAction                   pycram.robot_plans.actions.composite.tool_based.CuttingAction
  DetectAction                    pycram.robot_plans.actions.core.misc.DetectAction
  EfficientTransportAction        pycram.robot_plans.actions.composite.transporting.EfficientTransportAction
  FaceAtAction                    pycram.robot_plans.actions.composite.facing.FaceAtAction
  FollowToolCenterPointPathAction  pycram.robot_plans.actions.core.robot_body.FollowToolCenterPointPathAction
  GraspingAction                  pycram.robot_plans.actions.core.pick_up.GraspingAction
  LookAtAction                    pycram.robot_plans.actions.core.navigation.LookAtAction
  MixingAction                    pycram.robot_plans.actions.composite.tool_based.MixingAction
  MoveAndPickUpAction       

## Stage 2 — LLM Classifies Action Type

The LLM receives the full registry list and the NL instruction. It returns `ActionClassification` — an exact Python class name + confidence + reasoning.

In [6]:
from llmr.schemas.slots import ActionClassification
from llmr.reasoning.slot_filler import _CLASSIFIER_SYSTEM

# ── Build the prompt the LLM will receive ─────────────────────────────────────
class_list = "\n".join(f"  - {name}" for name in sorted(action_registry))
system_prompt = _CLASSIFIER_SYSTEM.format(action_classes=class_list)

print("=== System prompt sent to LLM ===")
print(system_prompt)
print(f"\n=== User message ===")
print(f"Instruction: {INSTRUCTION!r}")

=== System prompt sent to LLM ===
You are a robot action classifier.

Given a natural-language instruction, identify which robot action class it
corresponds to from the list of available action classes below.

Available action classes and schema summaries:
  - CarryAction
  - CloseAction
  - CuttingAction
  - DetectAction
  - EfficientTransportAction
  - FaceAtAction
  - FollowToolCenterPointPathAction
  - GraspingAction
  - LookAtAction
  - MixingAction
  - MoveAndPickUpAction
  - MoveAndPlaceAction
  - MoveTorsoAction
  - NavigateAction
  - OpenAction
  - ParkArmsAction
  - PickAndPlaceAction
  - PickUpAction
  - PlaceAction
  - PouringAction
  - ReachAction
  - SearchAction
  - SetGripperAction
  - TransportAction

Return the EXACT Python class name (e.g. "PickUpAction" not "pick up action").
Return structured JSON.


=== User message ===
Instruction: 'pick up the milk from the table'


In [7]:
# ── Actual LLM classification call ────────────────────────────────────────────
structured_llm = llm.with_structured_output(ActionClassification)

classification: ActionClassification = structured_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": INSTRUCTION},
])

print("=== LLM classification response ===")
print(f"  action_type : {classification.action_type}")
print(f"  confidence  : {classification.confidence:.2f}")
print(f"  reasoning   : {classification.reasoning}")

action_cls = action_registry.get(classification.action_type)
print(f"\nResolved class: {action_cls}")
assert action_cls is not None, f"Unknown action type: {classification.action_type!r}"

# Expected:
#   action_type : PickUpAction
#   confidence  : 1.00
#   reasoning   : "'pick up' maps directly to PickUpAction."
#   Resolved class: <class 'pycram.robot_plans.actions.core.PickUpAction'>

=== LLM classification response ===
  action_type : PickUpAction
  confidence  : 0.95
  reasoning   : The instruction specifies to 'pick up' an item (milk) from a location (the table), which directly corresponds to the PickUpAction class.

Resolved class: <class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>


## Stage 3 — Build Fully-Underspecified Match

`_fully_underspecified()` introspects the action class's dataclass fields and builds a `Match` with every field set to `...` (Ellipsis = free slot).

In [8]:
import dataclasses
from llmr.factory import _fully_underspecified, _get_settable_fields

fields = _get_settable_fields(action_cls)
print(f"Settable fields on {action_cls.__name__}:")
for f in fields:
    ftype = action_cls.__dataclass_fields__[f].type if dataclasses.is_dataclass(action_cls) else "?"
    print(f"  {f:30s}  type: {ftype}")

match = _fully_underspecified(action_cls)

print(f"\nMatch object : {match}")
print(f"Matches with variables ({len(list(match.matches_with_variables))}):")
for attr in match.matches_with_variables:
    print(f"  field={attr.name_from_variable_access_path:25s}  "
          f"value={attr.assigned_variable._value_!r:10}  "
          f"type={attr.assigned_variable._type_}")

Settable fields on PickUpAction:
  object_designator               type: Body
  arm                             type: Arms
  grasp_description               type: GraspDescription

Match object : Match(PickUpAction)
Matches with variables (3):
  field=PickUpAction.object_designator  value=Ellipsis    type=<class 'semantic_digital_twin.world_description.world_entity.Body'>
  field=PickUpAction.arm           value=Ellipsis    type=<enum 'Arms'>
  field=PickUpAction.grasp_description  value=Ellipsis    type=<class 'pycram.datastructures.grasp.GraspDescription'>


## Stage 3b — Introspect Action Schema with PycramIntrospector

`PycramIntrospector.introspect()` reads the pycram action dataclass and classifies
each field into a `FieldKind`.  This drives the **dynamic LLM prompt**.

| FieldKind | How resolved in `_evaluate` |
|-----------|------------|
| `ENTITY`   | EntityGrounder → Symbol instance (includes Manipulator, Camera) |
| `POSE`     | EntityGrounder → body.global_pose |
| `ENUM`     | string → enum member coercion |
| `COMPLEX`  | lazily introspected; sub-fields reconstructed from dotted SlotValues |
| `PRIMITIVE` | string coerced to bool/int/float/str |

Note: `_evaluate` classifies slots using krrood's **already-resolved** field types
(`attr_match.assigned_variable._type_`) — no full action-class re-introspection.


In [9]:
from llmr.pycram_bridge import PycramIntrospector, FieldKind

introspector = PycramIntrospector()
action_schema = introspector.introspect(action_cls)

print(f"=== ActionSchema for {action_schema.action_type} ===")
print(f"Docstring: {action_schema.docstring[:120]}...")
print(f"Fields ({len(action_schema.fields)}):")

for fspec in action_schema.fields:
    type_name = fspec.raw_type.__name__ if isinstance(fspec.raw_type, type) else str(fspec.raw_type)
    opt_str = " [optional]" if fspec.is_optional else " [required]"
    print(f"{fspec.name:30s}  kind={fspec.kind.value:10s}  type={type_name}{opt_str}")
    if fspec.docstring:
        print(f"    doc: {fspec.docstring[:100]}")
    if fspec.enum_members:
        print(f"    allowed values: {', '.join(fspec.enum_members)}")
    if fspec.sub_fields:
        print(f"    sub-fields:")
        for sub in fspec.sub_fields:
            sub_type = sub.raw_type.__name__ if isinstance(sub.raw_type, type) else str(sub.raw_type)
            members_note = f"  allowed: {', '.join(sub.enum_members)}" if sub.enum_members else ""
            print(f"      {sub.name:28s}  kind={sub.kind.value:10s}  type={sub_type}{members_note}")

# Expected for PickUpAction:
#   object_designator   kind=entity     type=Body        [required]
#   arm                 kind=enum       type=Arms        [required]
#   grasp_description   kind=complex    type=GraspDescription  [required]
#     sub-fields:
#       manipulator         kind=entity     type=Manipulator  (grounded from SymbolGraph)
#       grasp_type          kind=enum       type=GraspType
#       approach_direction  kind=enum       type=ApproachDirection


=== ActionSchema for PickUpAction ===
Docstring: Let the robot pick up an object....
Fields (3):
object_designator               kind=entity      type=Body [required]
    doc: Object designator_description describing the object that should be picked up
arm                             kind=enum        type=Arms [required]
    doc: The arm that should be used for pick up
    allowed values: LEFT, RIGHT, BOTH
grasp_description               kind=complex     type=GraspDescription [required]
    doc: The GraspDescription that should be used for picking up the object
    sub-fields:
      approach_direction            kind=enum        type=ApproachDirection  allowed: FRONT, BACK, RIGHT, LEFT
      vertical_alignment            kind=enum        type=VerticalAlignment  allowed: NoAlignment, TOP, BOTTOM
      manipulator                   kind=entity      type=Manipulator
      rotate_gripper                kind=primitive   type=bool
      manipulation_offset           kind=primitive   type=flo

## Stage 4 — Inspect Free vs Fixed Slots

This is what `LLMBackend._evaluate()` does internally to split the Match expression into free slots (LLM must fill) and fixed slots (already provided by the user).

In [10]:
# Strip 'ClassName.' prefix from name_from_variable_access_path
# (e.g. 'PickUpAction.arm' → 'arm') so downstream lookups work.
_short = lambda n: n.rsplit(".", 1)[-1] if "." in n else n

free_slots  = []   # (field_name, field_type) — LLM must fill these
fixed_slots = {}   # field_name → value     — already known
_full_name_map = {}  # short_name → full access path (for _get_mapped_variable_by_name)

for attr_match in match.matches_with_variables:
    field_name_raw = attr_match.name_from_variable_access_path
    field_name     = _short(field_name_raw)
    value          = attr_match.assigned_variable._value_
    field_type     = attr_match.assigned_variable._type_
    _full_name_map[field_name] = field_name_raw

    if isinstance(value, type(Ellipsis)):
        free_slots.append((field_name, field_type))
    else:
        fixed_slots[field_name] = value

print("FREE slots  (LLM will fill these):")
for name, typ in free_slots:
    print(f"  {name:30s}  type: {typ}")

print("FIXED slots (already known — passed as context to LLM):")
if fixed_slots:
    for name, val in fixed_slots.items():
        print(f"  {name:30s}  =  {val!r}")
else:
    print("  (none — user left everything open)")


FREE slots  (LLM will fill these):
  object_designator               type: <class 'semantic_digital_twin.world_description.world_entity.Body'>
  arm                             type: <enum 'Arms'>
  grasp_description               type: <class 'pycram.datastructures.grasp.GraspDescription'>
FIXED slots (already known — passed as context to LLM):
  (none — user left everything open)


## Stage 4b — All Free Slots go to the LLM

All free slots — including `Manipulator` and `Camera` fields — are sent to the slot filler. They are Symbol subclasses and are grounded from the SymbolGraph like any other entity (`Body`, `Region`, etc.).

There is no longer a separate "context injection" step.


In [11]:
field_specs = {f.name: f for f in action_schema.fields}

# All free slots go to the LLM — Manipulator/Camera are ENTITY, grounded from SymbolGraph
llm_free_slot_names = [name for name, _ in free_slots]

print("LLM-fillable fields (all free slots):")
for name in llm_free_slot_names:
    fspec = field_specs.get(name)
    kind_str = fspec.kind.value if fspec else "unknown"
    print(f"  {name:30s}  kind={kind_str}")


LLM-fillable fields (all free slots):
  object_designator               kind=entity
  arm                             kind=enum
  grasp_description               kind=complex


## Stage 5 — Serialize World via SymbolGraph

`serialize_world_from_symbol_graph(groundable_type)` builds the LLM world-context string from `SymbolGraph` — **no world object is passed**. This is the same pattern as `llmr_decoupled`'s `serialise_world_from_symbol_graph()`.

Filters out robot kinematic links by name-suffix heuristic (`_link`, `_frame`, `_joint`, etc.) rather than the old `/`-in-name trick that relied on SDT's `PrefixedName.__str__` format.

In [12]:
from llmr.world.serializer import serialize_world_from_symbol_graph

world_context = serialize_world_from_symbol_graph(groundable_type)
print(world_context)

# Expected (varies by world fixture):
#
# ## World State Summary
#
# Scene objects and surfaces: milk_bottle_1, cereal_box_1, kitchen_table, fridge_1, ...
#
# ## Semantic annotations
# Available types: FoodItem, Furniture, Graspable, HasSupportingSurface, ...
# Per body:
#   milk_bottle_1: FoodItem, Graspable
#   kitchen_table: Furniture, HasSupportingSurface
#   fridge_1:      Furniture, StorageUnit

## World State Summary

Scene objects and surfaces: base_footprint, map, odom_combined, apartment_root, furnitures_root, walls, windows, wall_coloksu_wall1, wall_coloksu_wall2, wall_coloksu_wall3, wall_coloksu_wall4, wardrobe, wardrobe_door_left, wardrobe_door_left_handle, wardrobe_door_right, wardrobe_door_right_handle, armchair, sofa, coffee_table, coffee_table_drawer, bedside_table, kitchen_root, side_A, side_B, cabinet1, cabinet1_door_top_left, handle_cab1_top_door, oven, cabinet1_drawer_middle, handle_cab1_drawer_mid, cabinet1_drawer_bottom, handle_cab1_drawer_bottom, cabinet1_coloksu_level4, cabinet1_coloksu_level5, cabinet2, cabinet2_door_out_fancy, cabinet2_door_left, handle_cab2_door, cabinet2_drawer_small, cabinet2_drawer_big, cabinet3, cabinet3_door_top_out_fancy, cabinet3_door_top_left, handle_cab3_door_top, cabinet3_door_bottom_out_fancy, cabinet3_door_bottom_left, handle_cab3_door_bottom, cabinet4, cabinet4_door_top, cabinet4_door_bottom, handle_cab4_door_bottom, cabinet5

## Stage 6a — Inspect Dynamic Prompt

`_build_dynamic_message()` generates the full user message for the LLM:

- **Entity slots** — role, Symbol type, pycram docstring; LLM returns `entity_description`
  (includes Symbol sub-fields like Manipulator inside complex types)
- **Enum params** — exact member names from the live Enum class; field docstring
- **Complex slots** — sub-fields expanded with dotted names and their valid values
- **Fixed slots** — already-resolved values the LLM must honour


In [13]:
from llmr.reasoning.slot_filler import (
    _introspect_free_slots,
    _build_dynamic_message,
    _SLOT_FILLER_SYSTEM,
)

introspected_specs = _introspect_free_slots(action_cls, llm_free_slot_names)

user_message = _build_dynamic_message(
    instruction      = INSTRUCTION,
    action_cls       = action_cls,
    free_slot_names  = llm_free_slot_names,
    fixed_slots      = fixed_slots,
    world_context    = world_context,
    field_specs      = introspected_specs,
)

print("=== System prompt (shared for all actions) ===")
print(_SLOT_FILLER_SYSTEM)
print("\n=== Dynamic user message (action-specific) ===")
print(user_message)

# Expected structure:
# Instruction: 'pick up the milk from the table'
# Action type: PickUpAction
# Action description: ...
#
# ── Entity slots (world objects) ──
#   - object_designator (Body): ...
#
# ── Parameter slots ──
#   - arm (allowed values: LEFT | RIGHT | BOTH) ...
#
# ── Complex slots ──
#   grasp_description (GraspDescription)
#     grasp_description.manipulator (Manipulator)      ← entity, LLM fills
#     grasp_description.grasp_type (TOP | FRONT | ...)


=== System prompt (shared for all actions) ===
You are a robot action parameter resolver with strong spatial and physical reasoning.

You receive:
  1. A natural-language instruction from a human operator.
  2. The target robot action type, its description, and all free parameter slots.
  3. Already-fixed slot values (do not change these).
  4. The current world state: scene objects, positions, and semantic annotations.

Your task: for every FREE slot, reason carefully and return a SlotValue.

────────────────────────────────────────────────────
ENTITY SLOTS  (objects / surfaces in the world)
────────────────────────────────────────────────────
Return a SlotValue with:
  - field_name  = the role name exactly as listed
  - entity_description populated:
      name           = noun phrase from the instruction (head noun only, no articles)
      semantic_type  = ontological type from world annotations (null if unknown)
      spatial_context = spatial qualifier from instruction ("on the tab

## Stage 6b — LLM Slot Filling (Core Reasoning)

`run_slot_filler()` sends the dynamic prompt to the LLM and returns an `ActionReasoningOutput` — a structured list of `SlotValue` objects.

For entity slots: each `SlotValue` carries an `EntityDescriptionSchema` (name + semantic_type + spatial_context + attributes) used by `EntityGrounder` in the next stage.

For enum/primitive slots: `SlotValue.value` is the resolved string (e.g. `"LEFT"`, `"TOP"`).

In [14]:
from llmr.reasoning.slot_filler import run_slot_filler
from llmr.schemas.slots import ActionReasoningOutput

output: ActionReasoningOutput = run_slot_filler(
    instruction     = INSTRUCTION,
    action_cls      = action_cls,
    free_slot_names = llm_free_slot_names,
    fixed_slots     = fixed_slots,
    world_context   = world_context,
    llm             = llm,
)

assert output is not None, "Slot filler returned None — check LLM connectivity"

print(f"=== ActionReasoningOutput ===")
print(f"action_type      : {output.action_type}")
print(f"overall_reasoning: {output.overall_reasoning}")
print(f"\nSlots ({len(output.slots)}):")

for sv in output.slots:
    print(f"\n  field_name : {sv.field_name}")
    print(f"  value      : {sv.value!r}")
    if sv.entity_description:
        ed = sv.entity_description
        print(f"  entity_description:")
        print(f"    name            = {ed.name!r}")
        print(f"    semantic_type   = {ed.semantic_type!r}")
        print(f"    spatial_context = {ed.spatial_context!r}")
        print(f"    attributes      = {ed.attributes}")
    print(f"  reasoning  : {sv.reasoning}")

# Expected:
#   field_name : object_designator
#   value      : 'milk_bottle_1'
#   entity_description:
#     name            = 'milk'
#     semantic_type   = 'Milk'
#     spatial_context = 'on the table'
#     attributes      = None
#   reasoning  : 'The milk is identified as a FoodItem on kitchen_table.'
#
#   field_name : arm
#   value      : 'RIGHT'
#   reasoning  : 'Right arm is free; object is on the right side of the robot.'
#
#   field_name : grasp_description.grasp_type
#   value      : 'TOP'
#   reasoning  : 'Approaching from above for a cylindrical bottle.'
#
#   field_name : grasp_description.approach_direction
#   value      : 'FRONT'
#   reasoning  : 'Object is directly in front of the robot.'

=== ActionReasoningOutput ===
action_type      : PickUpAction
overall_reasoning: The instruction to pick up the milk from the table has been analyzed, and the appropriate slots have been filled based on the identified object and typical grasping parameters.

Slots (7):

  field_name : object_designator
  value      : None
  entity_description:
    name            = 'milk'
    semantic_type   = 'Milk'
    spatial_context = 'on the table'
    attributes      = None
  reasoning  : The instruction specifies 'the milk', which corresponds to the object 'milk.stl' in the world state, identified as a Milk type.

  field_name : arm
  value      : 'RIGHT'
  reasoning  : Using the RIGHT arm is a common choice for picking up objects unless specified otherwise.

  field_name : grasp_description.approach_direction
  value      : 'FRONT'
  reasoning  : The approach direction is set to FRONT as it is typical to approach objects from the front for effective grasping.

  field_name : grasp_description.v

## Stage 7 — Entity Grounding via EntityGrounder

For ENTITY/POSE slots, `LLMBackend._evaluate()` calls `EntityGrounder.ground(entity_description)` which runs a two-tier search in `SymbolGraph`:

- **Tier 1** — annotation-based: resolves `semantic_type` string (e.g. `"Milk"`) to a Symbol subclass via `SymbolGraph.class_diagram`, then fetches instances via `.bodies`
- **Tier 2** — name-based: substring scan over all instances of `groundable_type`

The LLM never sees a `Body` object — it reasons over names and semantic types. Symbolic grounding happens here, deterministically.

In [15]:
# Imports from llmr — no krrood.llmr_decoupled references.
# EntityGrounder now accepts llmr's EntityDescriptionSchema directly;
# no role field or schema conversion needed.
from llmr.world.grounder import EntityGrounder
from llmr.pycram_bridge.introspector import FieldKind
from llmr.schemas.entities import EntityDescriptionSchema

grounder = EntityGrounder(groundable_type)
slot_by_name = {sv.field_name: sv for sv in output.slots}

print("=== Entity grounding results ===")

for fspec in action_schema.fields:
    if fspec.kind not in (FieldKind.ENTITY, FieldKind.POSE):
        continue
    sv = slot_by_name.get(fspec.name)
    if sv is None:
        print(f"  {fspec.name}: no SlotValue from LLM")
        continue

    ed = sv.entity_description
    if ed is not None:
        grounding_ed = ed  # already an EntityDescriptionSchema — pass directly
    else:
        grounding_ed = EntityDescriptionSchema(name=sv.value or fspec.name)

    result = grounder.ground(grounding_ed)
    print(f"  {fspec.name}:")
    print(f"    EntityDescription: name={grounding_ed.name!r}  semantic_type={grounding_ed.semantic_type!r}")
    if result.warning:
        print(f"    warning: {result.warning}")
    if result.bodies:
        from llmr.world.serializer import body_display_name
        print(f"    grounded to: {[body_display_name(b) for b in result.bodies]}")
        print(f"    using first: {body_display_name(result.bodies[0])}")
    else:
        print("    grounded to: NOTHING (entity not found in world)")

# Expected:
#   object_designator:
#     EntityDescription: name='milk'  semantic_type='Milk'
#     grounded to: ['milk.stl']
#     using first: milk.stl


=== Entity grounding results ===
  object_designator:
    EntityDescription: name='milk'  semantic_type='Milk'
    grounded to: ['milk.stl']
    using first: milk.stl


## Stage 8 — Write Resolved Values into Match → Construct Instance

This mirrors the exact pattern used by `ProbabilisticBackend` and `EntityQueryLanguageBackend._generate_raw_results()` in `krrood`.

`LLMBackend._evaluate()` resolves every free slot by `FieldKind`, writes into the Match variable graph, then constructs the concrete action instance.

In [16]:
from llmr.backend import (
    _resolve_entity_slot, _reconstruct_complex, _coerce_enum, _coerce_primitive,
    _UNRESOLVED,
)
from llmr.pycram_bridge import PycramIntrospector, FieldKind
from llmr.world.grounder import EntityGrounder

_intro = PycramIntrospector()
grounder = EntityGrounder(groundable_type)

print("=== Before assignment — Match variables ===")
for attr in match.matches_with_variables:
    name = _short(attr.name_from_variable_access_path)
    print(f"  {name:30s}  _value_ = {attr.assigned_variable._value_!r}")

slot_by_name = {sv.field_name: sv for sv in output.slots}
resolved_params = {}

print("=== Resolving each free slot (using krrood's resolved types) ===")
for field_name, field_type in free_slots:
    # field_type is already resolved by krrood — use it directly for classification
    kind = _intro._classify_type(field_type)

    if kind in (FieldKind.ENTITY, FieldKind.POSE):
        sv = slot_by_name.get(field_name)
        resolved = _resolve_entity_slot(sv, grounder, kind, field_name) if sv else _UNRESOLVED
        print(f"  {field_name:30s}  ENTITY  → {resolved}")

    elif kind == FieldKind.ENUM:
        sv = slot_by_name.get(field_name)
        resolved = _coerce_enum(sv.value, field_type) if sv and sv.value else _UNRESOLVED
        print(f"  {field_name:30s}  ENUM    → {resolved!r}")

    elif kind == FieldKind.COMPLEX:
        from llmr.pycram_bridge.introspector import FieldSpec
        sub_schema = _intro.introspect(field_type)
        fspec = FieldSpec(name=field_name, raw_type=field_type,
                         kind=kind, sub_fields=sub_schema.fields)
        resolved = _reconstruct_complex(field_name, fspec, slot_by_name,
                                        grounder, resolved_params)
        print(f"  {field_name:30s}  COMPLEX → {resolved!r}")

    else:
        sv = slot_by_name.get(field_name)
        resolved = _coerce_primitive(sv.value, field_type) if sv and sv.value else _UNRESOLVED
        print(f"  {field_name:30s}  PRIM    → {resolved!r}")

    if resolved is not _UNRESOLVED:
        resolved_params[field_name] = resolved
        full_name = _full_name_map.get(field_name, field_name)
        try:
            match._get_mapped_variable_by_name(full_name)._value_ = resolved
        except Exception as e:
            print(f"    WARNING: could not write '{field_name}': {e}")

match._update_kwargs_from_literal_values()
action_instance = match.construct_instance()

print(f"=== Constructed action instance ===")
print(f"  Type : {type(action_instance).__name__}")
print(f"  Value: {action_instance}")


=== Before assignment — Match variables ===
  object_designator               _value_ = Ellipsis
  arm                             _value_ = Ellipsis
  grasp_description               _value_ = Ellipsis
=== Resolving each free slot (using krrood's resolved types) ===
  object_designator               ENTITY  → Body(name=PrefixedName('None/milk.stl'), id=UUID('f5197525-a83f-44d6-ab2a-af7830a2cf40'), index=219)
  arm                             ENUM    → RIGHT
  grasp_description               COMPLEX → GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=ParallelGripper(name=PrefixedName('pr2/right_gripper'), id=UUID('5600af4a-bdc2-47a0-b642-8cf0a13ed87f'), root=Body(name=PrefixedName('pr2/r_gripper_palm_link'), id=UUID('2497cfd0-5382-4e53-af20-3358e0d3bdf0'), index=62), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('07ac2e8e-667c-461f

In [17]:
print(type(action_instance))
print(action_instance.__dict__.keys())

<class 'pycram.robot_plans.actions.core.pick_up.PickUpAction'>
dict_keys(['object_designator', 'arm', 'grasp_description'])


In [18]:
print(action_instance.object_designator, type(action_instance.object_designator))
print(action_instance.arm, type(action_instance.arm))
print(action_instance.grasp_description.__dict__.keys(), type(action_instance.grasp_description))
print(action_instance.grasp_description.approach_direction, type(action_instance.grasp_description.approach_direction))
print(action_instance.grasp_description.vertical_alignment, type(action_instance.grasp_description.vertical_alignment))

Body(name=PrefixedName('None/milk.stl'), id=UUID('f5197525-a83f-44d6-ab2a-af7830a2cf40'), index=219) <class 'semantic_digital_twin.world_description.world_entity.Body'>
RIGHT <enum 'Arms'>
dict_keys(['approach_direction', 'vertical_alignment', 'manipulator', 'rotate_gripper', 'manipulation_offset']) <class 'pycram.datastructures.grasp.GraspDescription'>
ApproachDirection.FRONT <enum 'ApproachDirection'>
VerticalAlignment.TOP <enum 'VerticalAlignment'>


In [19]:
action_instance.grasp_description.manipulator

ParallelGripper(name=PrefixedName('pr2/right_gripper'), id=UUID('5600af4a-bdc2-47a0-b642-8cf0a13ed87f'), root=Body(name=PrefixedName('pr2/r_gripper_palm_link'), id=UUID('2497cfd0-5382-4e53-af20-3358e0d3bdf0'), index=62), _robot=PR2(neck=Neck(name=PrefixedName('pr2/neck'), id=UUID('07ac2e8e-667c-461f-a4f8-1bcfbc787844'), root=Body(name=PrefixedName('pr2/head_pan_link'), id=UUID('4d25434f-29f8-4c00-97b0-e89769e0ed09'), index=21), _robot=..., joint_states=[], tip=Body(name=PrefixedName('pr2/head_tilt_link'), id=UUID('b353c675-c1ac-450f-9f64-9c985e57cb4e'), index=22), manipulator=None, sensors=[Camera(name=PrefixedName('pr2/wide_stereo_optical_frame'), id=UUID('49271df8-c951-4532-80dc-29e42e5c17ed'), root=Body(name=PrefixedName('pr2/wide_stereo_optical_frame'), id=UUID('0d9ad089-f2f5-4abc-9687-961b6d180e78'), index=29), _robot=..., joint_states=[], forward_facing_axis=Vector3(@1=0, [@1, @1, 1, @1]), field_of_view=FieldOfView(vertical_angle=0.75049, horizontal_angle=0.99483), minimal_height

pycram.robot_plans.actions.core.pick_up.PickUpAction

## Stage 9 — Role 2: `resolve_match()` — Pure Parameter Resolver

`resolve_match()` is the Role 2 entry point.  You already have a Match (action type
known, some slots fixed) and want the LLM to fill the remaining free parameters from
world context — no action classification, no Match construction.

`instruction` is optional: omit it when the action type and fixed slots carry the
intent.  Provide it when semantic grounding matters (e.g. which object, which surface).


In [21]:
from krrood.entity_query_language.query.match import Match
from llmr import resolve_match
from pycram.datastructures.enums import Arms

# Pre-fix arm=RIGHT — LLM fills object_designator + grasp_description only
power_match = Match(action_cls)(
    object_designator = ...,        # free — LLM grounds from world context
    arm               = Arms.RIGHT, # fixed — forced by caller
    grasp_description = ...,        # free — LLM reasons about grasp sub-fields
)

print("Match slots:")
for attr in power_match.matches_with_variables:
    name  = attr.name_from_variable_access_path
    value = attr.assigned_variable._value_
    status = "FREE (LLM fills)" if isinstance(value, type(Ellipsis)) else f"FIXED = {value!r}"
    print(f"  {name:30s}  {status}")

# resolve_match — Role 2 entry point, hides all krrood wiring
plan_node = resolve_match(
    match           = power_match,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
    # instruction omitted — action type + fixed slots carry the intent
)

print(f"\nPlan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"groundable : {context.query_backend.groundable_type.__name__}")


Match slots:
  PickUpAction.object_designator  FREE (LLM fills)
  PickUpAction.arm                FIXED = RIGHT
  PickUpAction.grasp_description  FREE (LLM fills)

Plan node  : PickUpAction
Backend    : LLMBackend
groundable : Body


## Stage 10 — Simple User API: `nl_plan()` end-to-end

`nl_plan()` runs all previous stages automatically. The user provides only the NL instruction and `groundable_type`. No Match, no action class, no backend — just language.

In [22]:
from llmr import nl_plan

# groundable_type is optional (defaults to Symbol).
# Pass Body for physical-body-scoped grounding.
plan_node = nl_plan(
    instruction     = INSTRUCTION,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
)

print(f"Plan node  : {plan_node}")
print(f"Backend    : {type(context.query_backend).__name__}")
print(f"Instruction: {context.query_backend.instruction!r}")
print(f"groundable : {context.query_backend.groundable_type.__name__}")


Plan node  : PickUpAction
Backend    : LLMBackend
Instruction: 'pick up the milk from the table'
groundable : Body


## Stage 11 — Execute

`plan.perform()` triggers `UnderspecifiedNode._perform()` → `context.query_backend.evaluate(match)` → `LLMBackend._evaluate()` → robot motion.

> **Note:** Run this cell only in a live simulation environment (Multiverse / Bullet). Comment out `plan_node.perform()` and wrap with `with simulated_robot:` if needed.

In [23]:
# from pycram.process_module import simulated_robot
# with simulated_robot:
#     plan_node.perform()

plan_node.perform()

# Expected console output:
# [INFO] LLMBackend._evaluate: resolving PickUpAction
# [INFO]   object_designator → Body(milk_bottle_1)
# [INFO]   arm               → Arms.RIGHT
# [INFO]   grasp_description → GraspDescription(grasp_type=TOP, approach=FRONT)
# [INFO] PickUpAction executing — grasping milk_bottle_1 with RIGHT arm

INFO:base.py::60 perform Performing action PickUpAction


ConditionNotSatisfied: Pre-Condition for Action 'PickUpAction' is not satisfied, following statements are false: ['pose_sequence_reachability_validator']

## Bonus — Multi-Step: `nl_sequential()`

Tests `TaskDecomposer` + `nl_plan()` chained together. Each step gets its own `LLMBackend` with a step-specific instruction and the shared `groundable_type`.

In [ ]:
from llmr.reasoning.decomposer import TaskDecomposer

MULTI_INSTRUCTION = "go to the table, pick up the milk, and put it in the fridge"

decomposer = TaskDecomposer(llm=llm)
decomposed = decomposer.decompose(MULTI_INSTRUCTION)

print("=== Decomposed steps ===")
for i, step in enumerate(decomposed.steps):
    deps = decomposed.dependencies.get(i, [])
    dep_str = f"  (depends on steps {deps})" if deps else ""
    print(f"  [{i}] {step}{dep_str}")

# Expected:
# [0] navigate to the table
# [1] pick up the milk from the table  (depends on steps [0])
# [2] place the milk in the fridge     (depends on steps [1])

In [ ]:
from llmr import nl_sequential

plan_nodes = nl_sequential(
    instruction     = MULTI_INSTRUCTION,
    context         = context,
    llm             = llm,
    groundable_type = groundable_type,
)

print(f"Built {len(plan_nodes)} plan nodes:")
for i, node in enumerate(plan_nodes):
    print(f"  [{i}] {node}")

print("\nExecuting...")
for i, node in enumerate(plan_nodes):
    print(f"\n--- Step {i} ---")
    node.perform()
